In [ ]:
import openeo
import time
from openeo.processes import quantiles, array_element, if_
from pathlib import Path

In [ ]:
# Remove connection to the backend and remove the refresh token 
#from openeo.rest.auth.config import RefreshTokenStore
#RefreshTokenStore().remove()

In [ ]:
connection = openeo.connect(url="openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

## Sentinel-2 mosaic from Sentinel-2 L2A

Create the Sentinel-2 mosaic from L2A data. Product description: 

https://dataspace.copernicus.eu/news/2024-2-27-exploring-new-frontier-sentinel-cloudless-mosaics-copernicus-data-space-ecosystem

https://documentation.dataspace.copernicus.eu/Data/SentinelMissions/Sentinel2.html#sentinel-2-level-3-quarterly-mosaics

In [ ]:
spatial_extent_west = {"west": 610950, "east": 674650, "south": 5143820, "north": 5206380, "crs": "EPSG:32632"} # Western South Tyrol glaciers
spatial_extent_east = {"west": 704100, "east": 747040, "south": 5195880, "north": 5219660, "crs": "EPSG:32632"} # Eastern South Tyrol glaciers

In [ ]:
years = [2018, 2019, 2021, 2022, 2024, 2025]

base_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S2_openEO/")

### 1) Generate the Sentinel-2 L2A mosaics and indices 

In [ ]:
BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]
BAD_SCL = [1, 3, 7, 8, 9, 10]   

def load_s2(spatial_extent, temporal_extent):
    cube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
        bands=BANDS + ["SCL"],
        max_cloud_cover=50,
    )
    return cube.resample_spatial(projection=32632, resolution=10, method="bilinear")

def cloud_mask(cube):
    scl = cube.band("SCL")
    m = scl == BAD_SCL[0]
    for v in BAD_SCL[1:]:
        m = m | (scl == v)
    return m

def create_s2_mosaic(spatial_extent, temporal_extent):
    cube = load_s2(spatial_extent, temporal_extent)
    mask = cloud_mask(cube)
    return (
        cube.mask(mask)
        .filter_bands(BANDS)
        .reduce_dimension(
            dimension="t",
            reducer=lambda x: array_element(quantiles(x, probabilities=[0.25]), index=0),
        )
    )

def glacier_mask(mosaic, ndsi_thr=0.4, ratio_thr=0.2):
    def classify(b):
        b03, b04, b11 = b["B03"], b["B04"], b["B11"]
        ndsi = (b03 - b11) / (b03 + b11)     # green vs SWIR
        ratio = (b04 - b11) / (b04 + b11)    # red vs SWIR
        return if_(ndsi > ndsi_thr, if_(ratio > ratio_thr, 1, 0), 0)
    return mosaic.reduce_dimension(dimension="bands", reducer=classify)

regions = {"west": spatial_extent_west, "east": spatial_extent_east}

for year in years:
    temporal_extent = [f"{year}-07-01T00:00:00Z", f"{year}-09-30T23:59:59Z"]

    results_folder = base_folder / str(year)
    results_folder.mkdir(parents=True, exist_ok=True)

    jobs = {}

    for name, extent in regions.items():
        result = glacier_mask(create_s2_mosaic(extent, temporal_extent)).save_result(format="GTiff")
        job = result.create_job(title=f"S2_glacier_{name}_{year}")
        job.start()
        jobs[name] = job
        print(f"{name} {year} started: {job.job_id}")
        time.sleep(10)

    for name, job in jobs.items():
        while job.status() not in ("finished", "error", "canceled"):
            time.sleep(30)
        if job.status() == "finished":
            out_file = results_folder / f"S2_glacier_{name}_{year}_results.tiff"
            job.get_results().download_file(out_file)
            print(f"Downloaded {out_file}")
        else:
            print(f"Job {name} {year} ended with status {job.status()}")